In [28]:
# =========================================================================
# CELL 1: IMPORT CÁC THƯ VIỆN CẦN THIẾT
# =========================================================================
import numpy as np
import pandas as pd
import os

# Thư viện tiền xử lý và đánh giá mô hình
from sklearn.metrics import roc_curve, auc, roc_auc_score, precision_recall_curve, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

# Thư viện Deep Learning (TensorFlow / Keras)
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.layers import Input, Dense, Concatenate, Dropout, BatchNormalization, Activation
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

print("Cell 1: Import thư viện thành công!")


Cell 1: Import thư viện thành công!


In [29]:
# =========================================================================
# CELL 2: KIỂM TRA CÁC FILE TRONG FOLDER INPUT CỦA KAGGLE
# =========================================================================
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/datasets/nambo2004/share-feature/shared_data_11000_dong.csv
/kaggle/input/datasets/nambo2004/benh-vien-b-private/private_B_10000_dong.csv
/kaggle/input/datasets/nambo2004/labels/labels_11000_dong.csv
/kaggle/input/datasets/nambo2004/benh-vien-a-private/private_A_1000_dong.csv


In [30]:
# =========================================================================
# CELL 3: TẢI 4 BẢNG DỮ LIỆU ĐÃ LÀM SẠCH VÀ HIỂN THỊ 10 DÒNG ĐẦU MỖI BẢNG
# =========================================================================

# 1. Bảng nhãn tử vong (labels)
df_labels = pd.read_csv('/kaggle/input/datasets/nambo2004/labels/labels_11000_dong.csv')
print(f"=== 1. BẢNG LABELS (Kích thước: {df_labels.shape[0]} dòng, {df_labels.shape[1]} cột) ===")
display(df_labels.head(10))

# 2. Bảng đặc trưng riêng Bệnh viện B (private B)
df_b = pd.read_csv('/kaggle/input/datasets/nambo2004/benh-vien-b-private/private_B_10000_dong.csv')
print(f"\n=== 2. BẢNG PRIVATE B (Kích thước: {df_b.shape[0]} dòng, {df_b.shape[1]} cột) ===")
display(df_b.head(10))

# 3. Bảng đặc trưng dùng chung (shared data)
df_shared = pd.read_csv('/kaggle/input/datasets/nambo2004/share-feature/shared_data_11000_dong.csv')
print(f"\n=== 3. BẢNG SHARED DATA (Kích thước: {df_shared.shape[0]} dòng, {df_shared.shape[1]} cột) ===")
display(df_shared.head(10))

# 4. Bảng đặc trưng riêng Bệnh viện A (private A)
df_a = pd.read_csv('/kaggle/input/datasets/nambo2004/benh-vien-a-private/private_A_1000_dong.csv')
print(f"\n=== 4. BẢNG PRIVATE A (Kích thước: {df_a.shape[0]} dòng, {df_a.shape[1]} cột) ===")
display(df_a.head(10))


=== 1. BẢNG LABELS (Kích thước: 11000 dòng, 2 cột) ===


,hadm_id,mortality
0,22600049,0
1,26774271,0
2,23181353,0
3,21453672,0
4,29740189,0
5,28991043,0
6,21610632,0
7,25929738,1
8,23795034,0
9,24401840,1



=== 2. BẢNG PRIVATE B (Kích thước: 10000 dòng, 15 cột) ===


,hadm_id,creatinine_max,bun_max,anion_gap,lactate_max,ph_min,potassium_mean,sodium_mean,chloride_mean,wbc_max,hemoglobin_min,platelets_min,inr_max,glucose_mean,bilirubin_max
0,27021321,1.7,64.0,18.0,1.4,7.33,3.700000,147.333333,106.666667,12.5,8.5,184.0,2.5,134.000000,0.3
1,20772846,0.8,13.0,17.0,1.2,7.34,4.450000,139.000000,105.000000,22.6,14.9,368.0,1.1,93.000000,0.4
2,21905572,1.6,13.0,15.0,3.2,7.23,3.800000,146.500000,121.000000,22.5,9.4,56.0,1.6,90.500000,0.2
3,28855134,3.4,27.0,17.0,1.1,7.13,3.666667,142.666667,114.000000,4.4,9.8,123.0,1.1,101.666667,0.7
4,28959049,0.8,14.0,12.0,1.5,7.31,4.450000,142.000000,102.500000,10.5,10.1,337.0,1.3,87.000000,0.8
5,22580364,2.0,50.0,21.0,1.7,7.38,4.050000,132.500000,94.500000,11.9,14.6,189.0,1.1,309.000000,0.5
6,22030954,1.1,20.0,28.0,8.6,7.27,4.400000,145.000000,103.000000,20.8,13.4,219.0,1.3,359.000000,0.5
7,28251507,4.2,99.0,17.0,1.3,7.26,4.625000,141.750000,102.500000,36.7,10.0,289.0,1.2,326.000000,0.6
8,21109754,1.6,27.0,12.0,1.8,7.21,4.300000,140.000000,110.000000,11.9,12.8,124.0,1.2,133.000000,0.4
9,23601441,0.9,23.0,11.0,1.2,7.36,4.200000,132.000000,100.000000,9.0,4.5,253.0,1.1,86.500000,0.3



=== 3. BẢNG SHARED DATA (Kích thước: 11000 dòng, 15 cột) ===


,hadm_id,gender,age,cardiovascular,neurological,pulmonary,diabetes,renal,liver,cancer,mental_substance,hem_metabolic,autoimmune,gcs_min,weight_kg
0,22600049,0,61,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,80.2
1,26774271,0,67,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,55.5
2,23181353,0,79,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,80.2
3,21453672,1,70,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,80.2
4,29740189,0,67,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,80.2
5,28991043,0,62,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,80.2
6,21610632,1,63,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,104.0
7,25929738,0,70,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,80.2
8,23795034,1,62,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,80.2
9,24401840,0,89,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,80.2



=== 4. BẢNG PRIVATE A (Kích thước: 1000 dòng, 10 cột) ===


,hadm_id,heart_rate_mean,sbp_mean,dbp_mean,mbp_mean,resp_rate_mean,temp_c_mean,spo2_mean,fio2_max,urine_output_24h
0,22600049,63.241379,121.800000,55.933333,73.466667,17.666667,36.650794,99.516129,50.0,3000.0
1,26774271,78.769231,107.814815,50.111111,68.222222,14.692308,36.444444,98.846154,100.0,3615.0
2,23181353,68.291667,123.212121,65.939394,90.666667,16.041667,37.629630,98.875000,40.0,1098.0
3,21453672,68.576923,101.741935,52.580645,68.129032,17.846154,36.896825,99.916667,100.0,3905.0
4,29740189,70.480000,110.923077,53.384615,70.923077,13.760000,36.777778,96.961538,100.0,4025.0
5,28991043,69.157895,111.500000,53.361111,67.361111,22.552632,37.194444,99.342105,40.0,1005.0
6,21610632,97.307692,109.533333,56.400000,71.033333,18.103448,36.694444,97.583333,100.0,1614.0
7,25929738,97.875000,108.090909,59.818182,73.818182,17.739130,37.277778,96.541667,50.0,850.0
8,23795034,80.285714,107.000000,60.448276,76.310345,14.428571,36.833333,97.178571,100.0,2060.0
9,24401840,76.772727,151.111111,43.611111,80.277778,22.363636,37.000000,100.000000,100.0,2015.0


In [31]:
# # =========================================================================
# # CELL 4: GHÉP DỮ LIỆU (MERGE) THEO 'hadm_id' VÀ HIỂN THỊ 10 DÒNG ĐẦU
# # =========================================================================

# # 1. Ghép dữ liệu cho Bệnh viện A (Private A + Shared + Labels)
# data_A = df_a.merge(df_shared, on='hadm_id').merge(df_labels, on='hadm_id')
# print("=" * 70)
# print(f"🏥 BỆNH VIỆN A SAU KHI GHÉP ĐẦY ĐỦ NHÃN VÀ BIẾN CHUNG:")
# print(f"• Số ca gốc: {len(df_a)} ca  -->  Sau khi ghép: {len(data_A)} ca")
# print(f"• Tổng số cột: {data_A.shape[1]} cột (Gồm hadm_id, Private A, Shared và Label)")
# print("=" * 70)
# display(data_A.head(10))

# # 2. Ghép dữ liệu cho Bệnh viện B (Private B + Shared + Labels)
# data_B = df_b.merge(df_shared, on='hadm_id').merge(df_labels, on='hadm_id')
# print("\n" + "=" * 70)
# print(f"🏥 BỆNH VIỆN B SAU KHI GHÉP ĐẦY ĐỦ NHÃN VÀ BIẾN CHUNG:")
# print(f"• Số ca gốc: {len(df_b)} ca  -->  Sau khi ghép: {len(data_B)} ca")
# print(f"• Tổng số cột: {data_B.shape[1]} cột (Gồm hadm_id, Private B, Shared và Label)")
# print("=" * 70)
# display(data_B.head(10))


In [32]:
# =========================================================================
# CELL 4: BỘ ĐẶC TRƯNG LÂM SÀNG 
# =========================================================================
import numpy as np

# 1. Ghép dữ liệu Bệnh viện A
data_A = df_a.merge(df_shared, on='hadm_id').merge(df_labels, on='hadm_id')

# Bộ biến sinh hiệu cấp cứu
data_A['shock_index'] = data_A['heart_rate_mean'] / (data_A['sbp_mean'] + 1e-5)
data_A['age_shock_index'] = data_A['age'] * data_A['shock_index']
data_A['temp_deviation'] = (data_A['temp_c_mean'] - 37.0).abs()
data_A['pulse_pressure'] = data_A['sbp_mean'] - data_A['dbp_mean']
data_A['map_drop_risk'] = np.maximum(0, 65.0 - data_A['mbp_mean'])
data_A['hypoxia_resp_risk'] = np.maximum(0, 95.0 - data_A['spo2_mean']) * data_A['resp_rate_mean']

# 2. Ghép dữ liệu Bệnh viện B
data_B = df_b.merge(df_shared, on='hadm_id').merge(df_labels, on='hadm_id')

# Bộ biến xét nghiệm đa tạng
data_B['bun_cr_ratio'] = data_B['bun_max'] / (data_B['creatinine_max'] + 1e-5)
data_B['acidosis_risk'] = data_B['anion_gap'] * data_B['lactate_max']
data_B['liver_coag_risk'] = data_B['bilirubin_max'] * data_B['inr_max']

print("Cell 4: Đã nạp xong bộ biến chuẩn")


Cell 4: Đã nạp xong bộ biến chuẩn


In [33]:
# =========================================================================
# CELL 5: CHIA TẬP TRAIN - VAL - TEST ĐỘC LẬP TRƯỚC KHI OVERSAMPLE
# BV A: 700 Train / 100 Validation / 200 Test
# BV B: 8.000 Train / 2.000 Test (hoặc 7.000 Train / 1.000 Val / 2.000 Test)
# =========================================================================
# Tách cột nhãn dự đoán (hỗ trợ cả tên 'mortality' hoặc 'mortality_1yr')
target_col = 'mortality' if 'mortality' in data_A.columns else 'mortality_1yr'

# --- 1. Tách Test độc lập (20%) ---
train_val_A, test_A = train_test_split(data_A, test_size=0.2, random_state=42, stratify=data_A[target_col])
train_val_B, test_B = train_test_split(data_B, test_size=0.2, random_state=42, stratify=data_B[target_col])

# --- 2. Tách Validation độc lập từ tập Train (12.5% của 800 ca = 100 ca Val, còn 700 ca Train) ---
train_A, val_A = train_test_split(train_val_A, test_size=0.125, random_state=42, stratify=train_val_A[target_col])
train_B, val_B = train_test_split(train_val_B, test_size=0.125, random_state=42, stratify=train_val_B[target_col])

print(f"🏥 Bệnh viện A: Train = {len(train_A)} | Val = {len(val_A)} | Test = {len(test_A)}")
print(f"🏥 Bệnh viện B: Train = {len(train_B)} | Val = {len(val_B)} | Test = {len(test_B)}")


🏥 Bệnh viện A: Train = 700 | Val = 100 | Test = 200
🏥 Bệnh viện B: Train = 7000 | Val = 1000 | Test = 2000


In [34]:
# # =========================================================================
# # CELL 6: TÁCH DANH SÁCH TÊN CỘT CHUNG VÀ CỘT RIÊNG
# # =========================================================================
# exclude_cols = ['hadm_id', 'mortality', 'mortality_1yr']

# common_cols = [c for c in df_shared.columns if c not in exclude_cols]
# spec_A_cols = [c for c in df_a.columns if c not in exclude_cols]
# spec_B_cols = [c for c in df_b.columns if c not in exclude_cols]

# print(f"📌 Số biến chung (Shared): {len(common_cols)}")
# print(f"📌 Số biến riêng A (Private A): {len(spec_A_cols)}")
# print(f"📌 Số biến riêng B (Private B): {len(spec_B_cols)}")


In [35]:
# =========================================================================
# CELL 6: CẬP NHẬT CỘT
# =========================================================================
exclude_cols = ['hadm_id', 'mortality', 'mortality_1yr']
common_cols = [c for c in df_shared.columns if c not in exclude_cols]
spec_A_cols = [c for c in data_A.columns if c not in exclude_cols and c not in common_cols]
spec_B_cols = [c for c in data_B.columns if c not in exclude_cols and c not in common_cols]
print(f"📌 Biến riêng A: {len(spec_A_cols)} biến | Biến riêng B: {len(spec_B_cols)} biến")


📌 Biến riêng A: 15 biến | Biến riêng B: 17 biến


In [36]:
# =========================================================================
# CELL 7: CHUẨN HÓA DỮ LIỆU (CHỈ FIT TRÊN TẬP TRAIN ĐỂ TRÁNH LEAKAGE)
# =========================================================================
# --- BỆNH VIỆN A ---
scaler_common_A = StandardScaler()
X_common_train_A = scaler_common_A.fit_transform(train_A[common_cols])
X_common_val_A   = scaler_common_A.transform(val_A[common_cols])
X_common_test_A  = scaler_common_A.transform(test_A[common_cols])

scaler_spec_A = StandardScaler()
X_spec_train_A = scaler_spec_A.fit_transform(train_A[spec_A_cols])
X_spec_val_A   = scaler_spec_A.transform(val_A[spec_A_cols])
X_spec_test_A  = scaler_spec_A.transform(test_A[spec_A_cols])

y_train_A = train_A[target_col].values
y_val_A   = val_A[target_col].values
y_test_A  = test_A[target_col].values

# --- BỆNH VIỆN B ---
scaler_common_B = StandardScaler()
X_common_train_B = scaler_common_B.fit_transform(train_B[common_cols])
X_common_val_B   = scaler_common_B.transform(val_B[common_cols])
X_common_test_B  = scaler_common_B.transform(test_B[common_cols])

scaler_spec_B = StandardScaler()
X_spec_train_B = scaler_spec_B.fit_transform(train_B[spec_B_cols])
X_spec_val_B   = scaler_spec_B.transform(val_B[spec_B_cols])
X_spec_test_B  = scaler_spec_B.transform(test_B[spec_B_cols])

y_train_B = train_B[target_col].values
y_val_B   = val_B[target_col].values
y_test_B  = test_B[target_col].values

print("✅ Đã chuẩn hóa toàn bộ dữ liệu Train, Val, Test thành công!")


✅ Đã chuẩn hóa toàn bộ dữ liệu Train, Val, Test thành công!


In [37]:
# =========================================================================
# CELL 8: OVERSAMPLING CHỈ TRÊN TẬP TRAIN A (TẬP VAL VÀ TEST KHÔNG CHẠM VÀO)
# =========================================================================
# Gộp X_common, X_spec và y của Train A để resample
train_A_temp = pd.DataFrame(X_common_train_A, columns=common_cols)
spec_df = pd.DataFrame(X_spec_train_A, columns=spec_A_cols)
train_A_temp = pd.concat([train_A_temp, spec_df], axis=1)
train_A_temp['target'] = y_train_A

# Mục tiêu: Tăng số mẫu Train A lên bằng đúng Train B (~7.000 dòng)
target_size = len(X_common_train_B)
oversampled_chunks = []
for label in [0, 1]:
    class_subset = train_A_temp[train_A_temp['target'] == label]
    class_fraction = len(class_subset) / len(train_A_temp)
    n_samples_needed = int(target_size * class_fraction)
    
    resampled_chunk = resample(
        class_subset, replace=True, n_samples=n_samples_needed, random_state=42
    )
    oversampled_chunks.append(resampled_chunk)

df_A_os = pd.concat(oversampled_chunks).sample(frac=1, random_state=42)

# Tách lại ra mảng numpy sau khi oversample
X_common_train_A_os = df_A_os[common_cols].values
X_spec_train_A_os   = df_A_os[spec_A_cols].values

# Mã hóa one-hot cho nhãn
y_train_encoded_A_os = to_categorical(df_A_os['target'].values)
y_val_encoded_A      = to_categorical(y_val_A)
y_train_encoded_B    = to_categorical(y_train_B)
y_val_encoded_B      = to_categorical(y_val_B)

# Khớp độ dài chính xác nếu có lệch 1-2 dòng do làm tròn
diff = len(X_common_train_B) - len(X_common_train_A_os)
if diff > 0:
    X_common_train_A_os = np.vstack([X_common_train_A_os, X_common_train_A_os[:diff]])
    X_spec_train_A_os   = np.vstack([X_spec_train_A_os, X_spec_train_A_os[:diff]])
    y_train_encoded_A_os = np.vstack([y_train_encoded_A_os, y_train_encoded_A_os[:diff]])

print(f"📊 Số lượng mẫu huấn luyện: Train A (sau Oversample) = {len(X_common_train_A_os)} | Train B = {len(X_common_train_B)}")
print(f"🛡️ Tập Validation A giữ nguyên: {len(X_common_val_A)} ca (Hoàn toàn không bị trùng lặp)")


📊 Số lượng mẫu huấn luyện: Train A (sau Oversample) = 7000 | Train B = 7000
🛡️ Tập Validation A giữ nguyên: 100 ca (Hoàn toàn không bị trùng lặp)


In [38]:
# =========================================================================
# CELL 9: DÒ TÌM SIÊU THAM SỐ TỐI ƯU CHO SHARED-PRIVATE
# =========================================================================
# 1. Khớp kích thước tập Validation của A với B để Keras không báo lỗi
val_idx_A = np.resize(np.arange(len(y_val_A)), len(y_val_B))
X_common_val_A_match = X_common_val_A[val_idx_A]
X_spec_val_A_match   = X_spec_val_A[val_idx_A]
y_val_encoded_A_match = y_val_encoded_A[val_idx_A]

learning_rates = [0.001, 0.0005]
weight_Bs = [0.2, 0.3, 0.5]

best_auc = 0.0
best_params = {}
l2_reg = regularizers.l2(1e-4)

print("=" * 60)
print("🚀 BẮT ĐẦU VÒNG LẶP DÒ TÌM THÔNG SỐ TỐI ƯU CHO SHARED-PRIVATE...")
print("=" * 60)

for lr in learning_rates:
    for wB in weight_Bs:
        tf.keras.backend.clear_session()
        tf.random.set_seed(42)

        # Inputs
        input_common_A = Input(shape=(len(common_cols),), name="X_common_A")
        input_common_B = Input(shape=(len(common_cols),), name="X_common_B")
        input_spec_A   = Input(shape=(len(spec_A_cols),), name="X_spec_A")
        input_spec_B   = Input(shape=(len(spec_B_cols),), name="X_spec_B")

        # Shared Encoder
        shared_layers = Sequential([
            Dense(64, kernel_regularizer=l2_reg), BatchNormalization(), Activation('relu'), Dropout(0.3),
            Dense(32, kernel_regularizer=l2_reg), BatchNormalization(), Activation('relu')
        ], name="Shared_Encoder")
        out_shared_A = shared_layers(input_common_A)
        out_shared_B = shared_layers(input_common_B)

        # Private Encoders
        priv_A = Sequential([
            Dense(32, activation='relu', kernel_regularizer=l2_reg),
            BatchNormalization(), Dropout(0.4),
            Dense(16, activation='relu', kernel_regularizer=l2_reg)
        ], name="Private_Encoder_A")
        out_priv_A = priv_A(input_spec_A)

        priv_B = Sequential([
            Dense(32, activation='relu', kernel_regularizer=l2_reg),
            BatchNormalization(), Dropout(0.3),
            Dense(16, activation='relu', kernel_regularizer=l2_reg)
        ], name="Private_Encoder_B")
        out_priv_B = priv_B(input_spec_B)

        # Heads
        def build_head(h_s, h_p, name):
            c = Concatenate()([h_s, h_p])
            x = Dense(16, activation='relu', kernel_regularizer=l2_reg)(c)
            x = Dropout(0.3)(x)
            return Dense(2, activation='softmax', name=name)(x)

        out_A = build_head(out_shared_A, out_priv_A, "y_A")
        out_B = build_head(out_shared_B, out_priv_B, "y_B")

        model_tune = Model(inputs=[input_common_A, input_spec_A, input_common_B, input_spec_B], outputs=[out_A, out_B])

        model_tune.compile(
            optimizer=Adam(learning_rate=lr),
            loss={"y_A": "categorical_crossentropy", "y_B": "categorical_crossentropy"},
            loss_weights={"y_A": 1.0, "y_B": wB}
        )

        early_stop = EarlyStopping(monitor='val_y_A_loss', mode='min', patience=5, restore_best_weights=True)

        # Huấn luyện với validation_data đã khớp kích thước
        model_tune.fit(
            x={"X_common_A": X_common_train_A_os, "X_spec_A": X_spec_train_A_os,
               "X_common_B": X_common_train_B,    "X_spec_B": X_spec_train_B},
            y={"y_A": y_train_encoded_A_os,       "y_B": y_train_encoded_B},
            validation_data=(
                {"X_common_A": X_common_val_A_match, "X_spec_A": X_spec_val_A_match,
                 "X_common_B": X_common_val_B,       "X_spec_B": X_spec_val_B},
                {"y_A": y_val_encoded_A_match,       "y_B": y_val_encoded_B}
            ),
            epochs=25,
            batch_size=64,
            callbacks=[early_stop],
            verbose=0
        )

        # Đánh giá trên đúng 100 ca Val gốc của A (không trùng lặp)
        val_pred = model_tune.predict(
            {"X_common_A": X_common_val_A, "X_spec_A": X_spec_val_A,
             "X_common_B": np.zeros_like(X_common_val_A), "X_spec_B": np.zeros((len(X_common_val_A), len(spec_B_cols)))},
            verbose=0
        )
        val_auc = roc_auc_score(y_val_A, val_pred[0][:, 1])
        print(f"🔄 Đã Test xong LR={lr}, Trọng số B={wB} ---> Val AUC của A = {val_auc:.4f}")

        if val_auc > best_auc:
            best_auc = val_auc
            best_params = {'Learning_Rate': lr, 'Trọng_số_B': wB}

print("\n" + "🔥" * 25)
print(f"🏆 CẤU HÌNH TỐI ƯU NHẤT CHO A: {best_params} (Val AUC: {best_auc:.4f})")
print("🔥" * 25)


🚀 BẮT ĐẦU VÒNG LẶP DÒ TÌM THÔNG SỐ TỐI ƯU CHO SHARED-PRIVATE...
🔄 Đã Test xong LR=0.001, Trọng số B=0.2 ---> Val AUC của A = 0.8975
🔄 Đã Test xong LR=0.001, Trọng số B=0.3 ---> Val AUC của A = 0.9017
🔄 Đã Test xong LR=0.001, Trọng số B=0.5 ---> Val AUC của A = 0.9096
🔄 Đã Test xong LR=0.0005, Trọng số B=0.2 ---> Val AUC của A = 0.8993
🔄 Đã Test xong LR=0.0005, Trọng số B=0.3 ---> Val AUC của A = 0.8704
🔄 Đã Test xong LR=0.0005, Trọng số B=0.5 ---> Val AUC của A = 0.8915

🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
🏆 CẤU HÌNH TỐI ƯU NHẤT CHO A: {'Learning_Rate': 0.001, 'Trọng_số_B': 0.5} (Val AUC: 0.9096)
🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥


In [ ]:
# =========================================================================
# CELL 10: HUẤN LUYỆN MODEL
# =========================================================================
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping

tf.keras.backend.clear_session()
tf.random.set_seed(999)

opt_lr = 0.001
opt_wB = 0.5
l2_reg = regularizers.l2(1e-4)

# 1. Inputs
in_common_A = layers.Input(shape=(len(common_cols),), name="X_common_A")
in_common_B = layers.Input(shape=(len(common_cols),), name="X_common_B")
in_spec_A   = layers.Input(shape=(len(spec_A_cols),), name="X_spec_A")
in_spec_B   = layers.Input(shape=(len(spec_B_cols),), name="X_spec_B")

# 2. Shared Encoder (Dropout 0.3 chuẩn)
shared_encoder = models.Sequential([
    layers.Dense(64, kernel_regularizer=l2_reg),
    layers.BatchNormalization(), layers.Activation('relu'), layers.Dropout(0.3),
    layers.Dense(32, kernel_regularizer=l2_reg),
    layers.BatchNormalization(), layers.Activation('relu')
], name="Shared_Encoder")
h_shared_A = shared_encoder(in_common_A)
h_shared_B = shared_encoder(in_common_B)

# 3. Private Encoders (Dropout 0.35 cho A để chống học vẹt tuyệt đối)
priv_A = models.Sequential([
    layers.Dense(32, activation='relu', kernel_regularizer=l2_reg),
    layers.BatchNormalization(), layers.Dropout(0.35),
    layers.Dense(16, activation='relu', kernel_regularizer=l2_reg)
], name="Private_Encoder_A")
h_priv_A = priv_A(in_spec_A)

priv_B = models.Sequential([
    layers.Dense(32, activation='relu', kernel_regularizer=l2_reg),
    layers.BatchNormalization(), layers.Dropout(0.3),
    layers.Dense(16, activation='relu', kernel_regularizer=l2_reg)
], name="Private_Encoder_B")
h_priv_B = priv_B(in_spec_B)

# 4. Wide & Deep Classification Heads
wide_deep_A = layers.Concatenate()([h_shared_A, h_priv_A, in_common_A, in_spec_A])
dense_A = layers.Dense(32, activation='relu', kernel_regularizer=l2_reg)(wide_deep_A)
dense_A = layers.Dropout(0.3)(dense_A)
dense_A = layers.Dense(16, activation='relu', kernel_regularizer=l2_reg)(dense_A)
out_A   = layers.Dense(2, activation='softmax', name="y_A")(dense_A)

wide_deep_B = layers.Concatenate()([h_shared_B, h_priv_B, in_common_B, in_spec_B])
dense_B = layers.Dense(32, activation='relu', kernel_regularizer=l2_reg)(wide_deep_B)
dense_B = layers.Dropout(0.3)(dense_B)
out_B   = layers.Dense(2, activation='softmax', name="y_B")(dense_B)

model_sp = models.Model(inputs=[in_common_A, in_spec_A, in_common_B, in_spec_B], outputs=[out_A, out_B])
model_sp.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=opt_lr),
    loss={"y_A": "categorical_crossentropy", "y_B": "categorical_crossentropy"},
    loss_weights={"y_A": 1.0, "y_B": opt_wB},
    metrics={"y_A": tf.keras.metrics.AUC(name="AUC"), "y_B": tf.keras.metrics.AUC(name="AUC")}
)

early_stop = EarlyStopping(monitor='val_y_A_AUC', mode='max', patience=6, restore_best_weights=True, verbose=1)

print("=" * 70)
print("🚀 ĐANG HUẤN LUYỆN LẠI VỚI DROPOUT CHUẨN 0.35...")
print("=" * 70)

history = model_sp.fit(
    x={"X_common_A": X_common_train_A_os, "X_spec_A": X_spec_train_A_os,
       "X_common_B": X_common_train_B,    "X_spec_B": X_spec_train_B},
    y={"y_A": y_train_encoded_A_os,       "y_B": y_train_encoded_B},
    validation_data=(
        {"X_common_A": X_common_val_A_match, "X_spec_A": X_spec_val_A_match,
         "X_common_B": X_common_val_B,       "X_spec_B": X_spec_val_B},
        {"y_A": y_val_encoded_A_match,       "y_B": y_val_encoded_B}
    ),
    epochs=25, batch_size=64, callbacks=[early_stop], verbose=1
)


🚀 ĐANG HUẤN LUYỆN LẠI VỚI DROPOUT CHUẨN 0.35...
Epoch 1/25
110/110 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - loss: 0.8526 - y_A_AUC: 0.8402 - y_A_loss: 0.5005 - y_B_AUC: 0.7194 - y_B_loss: 0.6513 - val_loss: 0.7000 - val_y_A_AUC: 0.9089 - val_y_A_loss: 0.3944 - val_y_B_AUC: 0.7880 - val_y_B_loss: 0.5584
Epoch 2/25
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.6978 - y_A_AUC: 0.9062 - y_A_loss: 0.3872 - y_B_AUC: 0.7819 - y_B_loss: 0.5686 - val_loss: 0.6432 - val_y_A_AUC: 0.9248 - val_y_A_loss: 0.3496 - val_y_B_AUC: 0.8072 - val_y_B_loss: 0.5345
Epoch 3/25
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.6442 - y_A_AUC: 0.9250 - y_A_loss: 0.3488 - y_B_AUC: 0.8041 - y_B_loss: 0.5391 - val_loss: 0.6159 - val_y_A_AUC: 0.9344 - val_y_A_loss: 0.3273 - val_y_B_AUC: 0.8156 - val_y_B_loss: 0.5243
Epoch 4/25
110/110 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.6129 - y_A_AUC: 0.9362 - y_A_loss: 0.3214 - y_B_AUC: 0.8103 - y_B_loss: 0.5312 - val_loss: 0.6094 - val_y_A_AUC: 0.9360 - val_y_A_loss: 0.3

In [40]:
# =========================================================================
# CELL 11: ĐÁNH GIÁ TRỰC TIẾP MÔ HÌNH TRÊN RAM (200 CA TEST BV A)
# =========================================================================
from sklearn.metrics import roc_curve, auc, average_precision_score, accuracy_score, confusion_matrix

# Dự đoán trực tiếp từ model_sp vừa train xong
test_preds = model_sp.predict(
    {"X_common_A": X_common_test_A, "X_spec_A": X_spec_test_A,
     "X_common_B": np.zeros_like(X_common_test_A), "X_spec_B": np.zeros((len(X_common_test_A), len(spec_B_cols)))},
    verbose=0
)
y_proba_A = test_preds[0][:, 1]

fpr, tpr, _ = roc_curve(y_test_A, y_proba_A)
test_auc_A = auc(fpr, tpr)
pr_auc_A = average_precision_score(y_test_A, y_proba_A)

y_pred_std = (y_proba_A >= 0.50).astype(int)
acc = accuracy_score(y_test_A, y_pred_std) * 100
correct = np.sum(y_pred_std == y_test_A)

cm = confusion_matrix(y_test_A, y_pred_std)
tn, fp, fn, tp = cm.ravel()
spec = (tn / (tn + fp)) * 100
rec = (tp / (tp + fn)) * 100

print("=" * 75)
print(f"🎯 KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH THỬ NGHIỆM (200 CA TEST BV A):")
print("=" * 75)
print(f"⭐ ROC-AUC Đạt Được         : {test_auc_A:.4f}")
print(f"⭐ PR-AUC (Precision-Recall) : {pr_auc_A:.4f}")
print(f"⭐ Độ chính xác (Accuracy)   : {acc:.2f}% ({correct}/200 ca đoán đúng)")
print(f"⭐ Độ đặc hiệu (Specificity) : {spec:.2f}%")
print(f"⭐ Độ nhạy tử vong (Recall)  : {rec:.2f}%")
print("=" * 75)


🎯 KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH THỬ NGHIỆM (200 CA TEST BV A):
⭐ ROC-AUC Đạt Được         : 0.8184
⭐ PR-AUC (Precision-Recall) : 0.5547
⭐ Độ chính xác (Accuracy)   : 82.00% (164/200 ca đoán đúng)
⭐ Độ đặc hiệu (Specificity) : 92.41%
⭐ Độ nhạy tử vong (Recall)  : 42.86%
